# SensorFM — downstream evaluation (Phase 4)

**Accelerator: GPU T4 x1** (P100 also fine). Settings: **Internet: On**. Secrets (Add-ons → Secrets): `GH_TOKEN`. Input: **Add Input → Your Datasets → `sensorfm-data`**. Also add the **03_pretrain** notebook output as input (for the checkpoint).

Budget ≈ 4.5 session-hours (incl. deployment).

In [ ]:
MODEL = 's20m'
TAG = MODEL

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ['GH_TOKEN'] = s.get_secret('GH_TOKEN')
try:
    os.environ['HF_TOKEN'] = s.get_secret('HF_TOKEN')
except Exception:
    print('no HF_TOKEN secret (optional)')

In [ ]:
%%bash
set -e
cd /kaggle/working
rm -rf sensorfm-src && git clone -q https://$GH_TOKEN@github.com/DhrumilPanchal10/sensorfm.git sensorfm-src
cd sensorfm-src && git log --oneline -1

In [ ]:
import glob
cks = sorted(glob.glob(f'/kaggle/input/*/runs/{MODEL}_main/checkpoints/step_*'))
assert cks, 'attach the 03_pretrain output as input'
CKPT = cks[-1]
print('using', CKPT)

In [ ]:
%%bash -s "$CKPT" "$MODEL" "$TAG"
set -e
cd /kaggle/working/sensorfm-src
cp -r /kaggle/input/*/results/eval/cmapss_baselines*.json results/eval/ 2>/dev/null || true
CKPT=$1 MODEL=$2 TAG=$3 bash kaggle/run_downstream.sh 2>&1 | grep -v Warning | tail -60

## Deployment (Tasks 5.1/5.2): ONNX + INT8 (CPU) and TensorRT FP16/INT8 (T4), ≈ 0.5 h

In [ ]:
%%bash -s "$CKPT" "$TAG"
set -e
cd /kaggle/working/sensorfm-src
CKPT=$1 TAG=$2 bash kaggle/run_deploy.sh 2>&1 | grep -v Warning | tail -30

In [ ]:
# Keep the saved output small: results/ and runs/ were copied to /kaggle/working already.
import shutil; shutil.rmtree('/kaggle/working/sensorfm-src', ignore_errors=True)
!du -sh /kaggle/working/* 2>/dev/null